# Aether Stage 2 — full training

This notebook performs the production Stage 2B run: **spoken question + text document → text answer**. It never sends the question transcript to Qwen. Run the smoke-test notebook successfully first.

Every new run writes directly to `MyDrive/aether-v2/stage2/runYYMMDD-HHMMSS`. Cache shards are shared under `MyDrive/aether-v2/stage2/cache/sqa5`; run checkpoints and logs are never overwritten.

In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")

import os, subprocess, sys
from pathlib import Path
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
assert os.environ["HF_TOKEN"], "Add HF_TOKEN in Colab Secrets and enable notebook access"

REPO_URL = "https://github.com/karl4th/aether-v3.git"
REPO_DIR = "/content/aether-v3"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", "stage2"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", "stage2", "--single-branch", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "stage2"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/stage2"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR, "huggingface_hub", "pytest", "ruff", "mypy"], check=True)
os.chdir(REPO_DIR)
print(subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

## 1. Configuration and private Stage 1 checkpoint

In [ ]:
CONFIG_PATH = "configs/stage2_train.yaml"
import torch
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer

from aether_v3.config import load_config
from aether_v3.models.aether_speech import AetherSpeechEncoder
from aether_v3.training.stage2_utils import load_stage1_encoder

cfg = load_config(CONFIG_PATH)
stage1_path = hf_hub_download(
    repo_id=cfg.stage2_train.stage1_repo_id,
    filename=cfg.stage2_train.stage1_filename,
    revision=cfg.stage2_train.stage1_revision,
    token=os.environ["HF_TOKEN"],
)
encoder = AetherSpeechEncoder(cfg.aether_speech)
checkpoint = load_stage1_encoder(stage1_path, encoder)
encoder.eval()
print("Stage 1 encoder loaded; checkpoint step:", checkpoint.get("step", "unknown"))

## 2. Prepare restartable SLUE-SQA-5 speech-state shards

This is restartable. Completed shards remain on Drive. Only spoken questions are encoded; document audio is ignored and `raw_document_text` becomes the normal Qwen text context.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer
from aether_v3.data.stage2_cache import build_slue_sqa5_shards
from aether_v3.models.mimi_wrapper import FrozenMimi

cache_root = Path(cfg.stage2_train.drive_root) / "cache" / "sqa5"
cache_root.mkdir(parents=True, exist_ok=True)
tokenizer = AutoTokenizer.from_pretrained(cfg.llm.model_id, revision=cfg.llm.revision)
mimi = FrozenMimi(cfg.mimi.pretrained_id, cfg.mimi.num_quantizers, device="cuda")
for role, split in (("train", cfg.stage2_data.train_split), ("validation", cfg.stage2_data.validation_split)):
    rows = load_dataset(cfg.stage2_data.dataset_id, cfg.stage2_data.dataset_config, split=split, streaming=True, token=os.environ["HF_TOKEN"], trust_remote_code=True)
    count = build_slue_sqa5_shards(rows, mimi, encoder, tokenizer, cache_root/role, role)
    print(role, count)
del mimi
torch.cuda.empty_cache()

## 3. Create an isolated run and train

To resume an existing run, set `cfg.stage2_train.resume_from` to its `last.pt` and set `run_dir` to that existing directory. Leave it unset to create a fresh run.

In [ ]:
from datetime import datetime
from pathlib import Path
from aether_v3.models.aether_speech_llm import AetherSpeechLLM
from aether_v3.training.stage2_utils import create_run_dir, load_stage1_encoder
from aether_v3.training.train_stage2 import run_stage2_training

if cfg.stage2_train.resume_from:
    run_dir = Path(cfg.stage2_train.resume_from).parent
else:
    run_dir = create_run_dir(cfg.stage2_train.drive_root)
model = AetherSpeechLLM(cfg.aether_speech, cfg.connector, cfg.llm, speech_frozen=True)
load_stage1_encoder(stage1_path, model.encoder)
print("Training output:", run_dir)
run_stage2_training(cfg, run_dir, cache_root/"train", cache_root/"validation", model=model)

## 4. Inspect results

In [ ]:
import json
rows = [json.loads(line) for line in open(run_dir/"log.jsonl")]
print("last records:")
for row in rows[-10:]: print(row)
print("checkpoints:")
for path in sorted(run_dir.glob("*.pt")): print(path.name)
print("periodic:", len(list((run_dir/"periodic").glob("*.pt"))))